In [ ]:
# import data tools
import pandas as pd
import altair as alt
import vl_convert

# load csv files
sales = pd.read_csv("Data Model - Pizza Sales.csv")

# set size order
size_order = ["S", "M", "L", "XL", "XXL"]
sales["pizza_size"] = pd.Categorical(sales["pizza_size"], categories=size_order, ordered=True)

# figure 2 
sales_small = sales.drop_duplicates(subset=["order_details_id"])

vis2 = (
    alt.Chart(sales_small)
    .mark_bar()
    .encode(
        x=alt.X("pizza_category:N", title="Category"),
        y=alt.Y("sum(total_price):Q", title="Total Revenue ($)"),
        color=alt.Color(
            "pizza_size:O",
            title="Size",
            scale=alt.Scale(scheme="yelloworangered")
        ),
        order=alt.Order("pizza_size:O")
    )
    .properties(width=500, height=350, title="Figure 2: Size contribution to total sales by category")
)

vis2.save("update_size_contribution.html")
print("Saved: update_size_contribution.html")

# save figure 2 as png
vis2.save("size_contribution_by_sales.png")
print("Saved: size_contribution_by_sales.png")

# figure 3: top pizzas by revenue or popularity
clean_sales = sales.drop_duplicates(subset=["order_details_id"])

popularity = (
    clean_sales.groupby("pizza_name")
    .size()
    .reset_index(name="order_count")
)

revenue = (
    clean_sales.groupby("pizza_name")
    .agg({"total_price": "sum"})
    .reset_index()
)

metrics = revenue.merge(popularity, on="pizza_name", how="left")

# dropdown parameter
sort_choice = alt.param(
    name="sort_by",
    bind=alt.binding_select(options=["Revenue", "Popularity"], name="Sort by: "),
    value="Revenue"
)

# final chart
vis3 = (
    alt.Chart(metrics)
    .add_params(sort_choice)
    .transform_calculate(
        metric="sort_by === 'Revenue' ? datum.total_price : datum.order_count"
    )
    .transform_window(
        rank="rank(metric)",
        sort=[{"field": "metric", "order": "descending"}]
    )
    .transform_filter("datum.rank <= 5")
    .mark_bar(color="#D95F02")
    .encode(
        x=alt.X("total_price:Q", title="Revenue ($)"),
        y=alt.Y("pizza_name:N", sort="-x", title="Pizza Name"),
        tooltip=["pizza_name", "total_price", "order_count"]
    )
    .properties(width=500, height=350, title="Figure 3: Top 5 Pizzas by Revenue")
)

vis3.save("update_top_pizza_by_revenue.html")
print("Saved: update_top_pizza_by_revenue.html")

# save figure 3 as png
vis3.save("top_pizza_by_revenue.png")
print("Saved: top_pizza_by_revenue.png")

Saved: update_size_contribution.html
Saved: size_contribution_by_sales.png
Saved: update_top_pizza_by_revenue.html
Saved: top_pizza_by_revenue.png
